# 02 — Exploratory Data Analysis

## Purpose
Five presentation-ready figures that tell the churn story from five angles:
contract structure, pricing dynamics, early friction, tenure risk, and
cross-feature correlations. Each figure is designed to stand alone in a
slide deck or executive briefing.

## Key Findings at a Glance
| Theme                        | Signal                                                             |
|------------------------------|--------------------------------------------------------------------|
| Contract type                | Month-to-month customers churn at ~2.5× the rate of two-year contracts |
| Internet service             | Fiber customers churn more than DSL (competitive alternatives)    |
| Promo pricing                | Customers on promotion churn more — they leave when the promo ends |
| Early friction (60-day)      | Unresolved issues nearly double the churn rate vs. resolved ones   |
| Tenure                       | First 12 months carry the highest churn risk                       |
| Payment method               | Manual-pay customers churn at ~2× the rate of autopay customers    |

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

STEEL_BLUE_900  = "#003865"
STEEL_BLUE_500  = "#0072CE"
WHITE_BG        = "#FFFFFF"
POSITIVE_GREEN  = "#2E7D32"
NEGATIVE_RED    = "#C62828"
BLUE_VARIATIONS = ["#003865", "#005696", "#0072CE", "#4192D9", "#82B1FF"]

plt.rcParams.update({
    "figure.facecolor": WHITE_BG, "axes.facecolor": WHITE_BG,
    "text.color": STEEL_BLUE_900, "axes.labelcolor": STEEL_BLUE_900,
    "xtick.color": STEEL_BLUE_900, "ytick.color": STEEL_BLUE_900,
    "font.family": "sans-serif",
})

df = pd.read_csv("../data/telco_churn_225k_v120.csv")
df["billing_call_resolved"] = df["billing_call_resolved"].astype("Int64")
df["tech_issue_resolved"]   = df["tech_issue_resolved"].astype("Int64")

CHURN_COL = "churn"
churn_rate = lambda s: s.mean() * 100

def annotate_bars(ax):
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.1f}%",
                    (p.get_x() + p.get_width() / 2.0, p.get_height()),
                    ha="center", va="center", xytext=(0, 9),
                    textcoords="offset points",
                    color=STEEL_BLUE_900, fontweight="bold")

print(f"Loaded {len(df):,} rows  |  overall churn rate: {df[CHURN_COL].mean():.1%}")

## Figure 1 — Core Business Drivers

**Contract type** is the single most powerful segmentation lever. Month-to-month
customers churn at roughly 2.5× the rate of two-year subscribers — consistent with
the widely-observed "contract lock-in" effect in subscription businesses.

**Fiber internet** customers churn more than DSL. This counter-intuitive finding
reflects that fiber markets are more competitive; customers have viable alternatives
and are more price-sensitive.

**Sales channel** shows that third-party retailers and agent call centers produce
higher-risk customers — likely because those channels attract more price-shoppers
or execute lower-quality onboarding.

In [ ]:
fig1, axes = plt.subplots(1, 3, figsize=(18, 6))
fig1.suptitle("Figure 1: Core Business Drivers", fontsize=16, fontweight="bold", color=STEEL_BLUE_900)

sns.barplot(data=df.groupby("contract_type")[CHURN_COL].apply(churn_rate).reset_index(),
            x="contract_type", y=CHURN_COL, palette=BLUE_VARIATIONS[:3], ax=axes[0])
axes[0].set_title("Churn by Contract Type", fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[0])

sns.barplot(data=df.groupby("internet_service")[CHURN_COL].apply(churn_rate).reset_index(),
            x="internet_service", y=CHURN_COL, palette=BLUE_VARIATIONS[:3], ax=axes[1])
axes[1].set_title("Churn by Internet Service", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[1])

sns.barplot(data=df.groupby("sales_channel")[CHURN_COL].apply(churn_rate).reset_index(),
            x="sales_channel", y=CHURN_COL, palette=BLUE_VARIATIONS[:4], ax=axes[2])
axes[2].set_title("Churn by Sales Channel", fontweight="bold")
axes[2].set_ylabel("Churn Rate (%)")
axes[2].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[2].tick_params(axis="x", rotation=15)
annotate_bars(axes[2])

plt.tight_layout()
plt.savefig("../reports/Presentation_Fig1_Drivers.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 2 — Promo & Pricing Dynamics

Promotional pricing is a double-edged sword. Customers currently **on promo churn
at a higher rate** than full-rate customers. The mechanism: promos attract
price-sensitive customers who leave once the discount expires — a classic adverse
selection problem.

Notably, the highest discount tiers (>35%) do **not** produce the lowest churn —
if anything, they attract the most price-sensitive subscribers who are least loyal
to the brand long-term.

In [ ]:
fig2, axes = plt.subplots(1, 2, figsize=(14, 6))
fig2.suptitle("Figure 2: Promo & Pricing Dynamics", fontsize=16, fontweight="bold", color=STEEL_BLUE_900)

df["promo_status"] = np.where(df["is_on_promo"] == 1, "On Promo", "Full Rate")
sns.barplot(data=df.groupby("promo_status")[CHURN_COL].apply(churn_rate).reset_index(),
            x="promo_status", y=CHURN_COL,
            palette=[STEEL_BLUE_900, STEEL_BLUE_500], ax=axes[0])
axes[0].set_title("Churn: On Promo vs. Full Rate", fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[0])

promo_df = df[df["is_on_promo"] == 1].copy()
promo_df["discount_tier"] = pd.cut(promo_df["promo_discount_pct"],
    bins=[0, 0.15, 0.25, 0.35, 1.0],
    labels=["Low (≤15%)", "Mid (16–25%)", "High (26–35%)", "Very High (>35%)"])
sns.barplot(data=promo_df.groupby("discount_tier", observed=True)[CHURN_COL].apply(churn_rate).reset_index(),
            x="discount_tier", y=CHURN_COL, palette=BLUE_VARIATIONS[:4], ax=axes[1])
axes[1].set_title("Churn by Discount Tier (Promo Customers Only)", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
axes[1].tick_params(axis="x", rotation=10)
annotate_bars(axes[1])

plt.tight_layout()
plt.savefig("../reports/Presentation_Fig2_Pricing.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 3 — First 60-Day Friction Impact

Early friction is the most **operationally actionable** driver in this analysis.
A billing or tech issue in the first 60 days, **if left unresolved**, nearly doubles
the churn rate compared to customers whose issues were resolved.

The implication is direct: the investment isn't in *preventing* all service issues
(some are unavoidable), but in ensuring **rapid resolution** when they occur. This
frames the retention ROI calculation in notebook 05.

In [ ]:
fig3, axes = plt.subplots(1, 2, figsize=(14, 6))
fig3.suptitle("Figure 3: First 60-Day Friction Impact", fontsize=16, fontweight="bold", color=STEEL_BLUE_900)

billing_df = df[df["first_60d_billing_call"] == 1].copy()
billing_df["resolution"] = billing_df["billing_call_resolved"].map({1: "Resolved", 0: "Unresolved"})
sns.barplot(data=billing_df.groupby("resolution")[CHURN_COL].apply(churn_rate).reset_index(),
            x="resolution", y=CHURN_COL,
            palette=[POSITIVE_GREEN, NEGATIVE_RED], ax=axes[0])
axes[0].set_title("Billing Issue Resolution vs. Churn", fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[0])

tech_df = df[df["first_60d_tech_call"] == 1].copy()
tech_df["resolution"] = tech_df["tech_issue_resolved"].map({1: "Resolved", 0: "Unresolved"})
sns.barplot(data=tech_df.groupby("resolution")[CHURN_COL].apply(churn_rate).reset_index(),
            x="resolution", y=CHURN_COL,
            palette=[POSITIVE_GREEN, NEGATIVE_RED], ax=axes[1])
axes[1].set_title("Tech Issue Resolution vs. Churn", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[1])

plt.tight_layout()
plt.savefig("../reports/Presentation_Fig3_Friction.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 4 — Tenure Risk & Payment Behavior

**Tenure bucketing** reveals the classic "bathtub curve" of churn risk: new
subscribers (0–12 months) are by far the most vulnerable. This is the onboarding
window where the customer hasn't yet built switching costs or habitual usage.
After ~24 months, churn risk stabilizes significantly.

**Payment method** is a strong proxy for engagement and commitment. Autopay
customers have materially lower churn — they've made an active decision to
automate payments, which correlates with intent to stay.

In [ ]:
fig4, axes = plt.subplots(1, 2, figsize=(14, 6))
fig4.suptitle("Figure 4: Tenure & Payment Behavior", fontsize=16, fontweight="bold", color=STEEL_BLUE_900)

df["tenure_bucket"] = pd.cut(df["tenure_months"], bins=[0, 12, 24, 36, 48, 72],
                              labels=["0–12 mo", "13–24 mo", "25–36 mo", "37–48 mo", "49–72 mo"])
sns.barplot(data=df.groupby("tenure_bucket", observed=True)[CHURN_COL].apply(churn_rate).reset_index(),
            x="tenure_bucket", y=CHURN_COL, palette=BLUE_VARIATIONS[:5], ax=axes[0])
axes[0].set_title("Churn by Tenure Bucket", fontweight="bold")
axes[0].set_ylabel("Churn Rate (%)")
axes[0].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[0])

df["pay_method_label"] = np.where(df["is_autopay"] == 1, "Autopay", "Manual Pay")
sns.barplot(data=df.groupby("pay_method_label")[CHURN_COL].apply(churn_rate).reset_index(),
            x="pay_method_label", y=CHURN_COL,
            palette=[POSITIVE_GREEN, NEGATIVE_RED], ax=axes[1])
axes[1].set_title("Churn by Payment Method", fontweight="bold")
axes[1].set_ylabel("Churn Rate (%)")
axes[1].yaxis.set_major_formatter(mtick.PercentFormatter())
annotate_bars(axes[1])

plt.tight_layout()
plt.savefig("../reports/Presentation_Fig4_TenureRisk.png", dpi=150, bbox_inches="tight")
plt.show()

## Figure 5 — Correlation Heatmap

The heatmap quantifies what the bar charts show directionally. Key correlations
with the churn target:

- **`churn_score`** has the highest correlation with churn (by construction — it's
  the synthetic driver). This column is excluded from all ML models.
- **`contract_type`** (encoded via `billing_risk_score`) is the strongest *observable* signal.
- **`tenure_months`** is negatively correlated — longer tenure = lower churn.
- **`is_autopay`** is negatively correlated — autopay customers churn less.

In [ ]:
fig5, ax = plt.subplots(figsize=(12, 10))
numeric_cols = ["tenure_months", "monthly_charges", "monthly_charges_billed", "total_charges",
                "is_on_promo", "promo_discount_pct", "first_60d_billing_call",
                "is_autopay", "billing_risk_score", "churn_score"]
corr_df = df[numeric_cols].copy()
corr_df["churned"] = df[CHURN_COL].astype(int)

sns.heatmap(corr_df.corr(), annot=True, fmt=".2f", cmap="Blues", center=0, ax=ax,
            annot_kws={"color": STEEL_BLUE_900})
ax.set_title("Figure 5: Correlation Heatmap", fontsize=16, fontweight="bold", color=STEEL_BLUE_900)
plt.tight_layout()
plt.savefig("../reports/Presentation_Fig5_Correlation.png", dpi=150, bbox_inches="tight")
plt.show()